## Segmentation Mask Visualization and Contour Export

This notebook demonstrates how to work with segmentation masks generated by **Cellpose**, **StarDist**, or any other method (e.g. Baysor) in the ISS postprocessing pipeline. The visualization function supports two image modes via `input_image_type = "stitched" | "retiled"`.

---

### Workflow

1. **Load Segmentation Mask**  
The segmentation mask is loaded from a sparse `.npz` file and converted into a dense label image.

There are two ways the mask is resolved:

- **Default lookup (automatic):**  
`<input_dir>/<region>/postprocessing/segmentation/`  
Examples:  
`R1_cellpose_stitched_expanded.npz`  
`R1_stardist_retiled_expanded.npz`

- **Explicit file (recommended for flexibility):**  
`segmentation_file="/path/to/your_mask.npz"`  
This can be **any segmentation method** (Cellpose, StarDist, Baysor, custom).

If `segmentation_file` is provided, it overrides all automatic lookup logic.

---

2. **Load Corresponding Raw Image**  
The raw **DAPI channel** is loaded from preprocessing.

- **Stitched mode:**  
`<input_dir>/<region>/preprocessing/Cycle1/3_stitched/Cycle1_ch{DAPI_ch}.tif`

- **Retiled mode:**  
Reconstructed automatically from:  
`<input_dir>/<region>/preprocessing/Cycle1/4_retiled/Cycle1_s*_ch{DAPI_ch}.tif`  
using:  
`Cycle1_retiled_coords.csv`

---

3. **Optional Crop (Zoom-in)**  
A region can be selected for inspection using:  
`crop_coords = (y_start, y_end, x_start, x_end)`  

Example:  
`(4000, 8000, 4000, 8000)`  

If `None`, the full image is used.

---

4. **Normalize and Brighten Image**  
The image is scaled to `[0, 1]` and slightly brightened for better visibility.

---

5. **Overlay Segmentation Boundaries**  
Segmentation boundaries are drawn on top of the image using `skimage.segmentation.mark_boundaries`. Visualization is sized to fit typical screens (≈6×6 inches, moderate DPI).

---

6. **Generate Contour Mask**  
A binary contour mask is created:  
- Boundaries extracted with `find_boundaries(mode="outer")`  
- Thickened using `binary_dilation`  
- Converted to 8-bit (white contours on black)

---

7. **Save Contour Mask**  
The contour mask is saved to:  
`<output_dir>/<region>/postprocessing/segmentation/`  

Where:  
- `output_dir = output_dir_prefix` if provided  
- otherwise defaults to `<input_dir>/<region>/postprocessing/segmentation/`

Filename format:  
`<region>_<segmentation_method>_<input_image_type>_contour_mask.tif`  

Examples:  
`R1_cellpose_stitched_contour_mask.tif`  
`R1_stardist_retiled_contour_mask.tif`  
`R1_baysor_stitched_contour_mask.tif`

---

### Notes

- `segmentation_method` is used as a **label/tag**, not strictly tied to the algorithm.  
- If `segmentation_file` is provided, a warning is shown if the method name does not match the filename.  
- The segmentation mask **must match the raw image shape**, otherwise an error is raised.  
- `input_dir` is always used to locate the **raw image**, even if the segmentation comes from elsewhere.

---

### Tip

If your segmentation is not found automatically or comes from another pipeline, always use:  
`segmentation_file="full/path/to/mask.npz"`  

This is the most robust and flexible option.

### Imports

In [ ]:
import ISS_postprocessing.segmentation as SEG

### Inspect Segmentation

Load a segmentation mask, overlay it on the DAPI image, and visualize the result.

#### Segmentation input

- **Default:**
  `<input_dir>/<region>/postprocessing/segmentation/`

- **Recommended (more robust):**
  `segmentation_file="/path/to/mask.npz"`  
  → works with **any method** (Cellpose, StarDist, Baysor, etc.)

If `segmentation_file` is provided, it overrides default lookup.

#### Expected filenames (default lookup only)

- Stitched:  
  `R1_cellpose_stitched_expanded.npz`  
  `R1_stardist_stitched_expanded.npz`

- Retiled:  
  `R1_cellpose_retiled_expanded.npz`  
  `R1_stardist_retiled_expanded.npz`


#### Raw image (for overlay)

- Stitched:  
  `<input_dir>/<region>/preprocessing/Cycle1/3_stitched/Cycle1_ch{DAPI_ch}.tif`

- Retiled:  
  reconstructed from:  
  `<input_dir>/<region>/preprocessing/Cycle1/4_retiled/*.tif`  
  + `Cycle1_retiled_coords.csv`


#### Input Parameters

`input_dir` (str): Root folder containing region folders (`R1`, `R2`, …)  
`region` (str): Single region (e.g. `"R1"`)  
`segmentation_method` (str): Label for default lookup and output naming  
- must match filename if using default lookup  
- can be any label if using `segmentation_file` (e.g. `"baysor"`)  

`segmentation_file` (str | Path | None): Explicit `.npz` mask  
- overrides all lookup logic  
- supports any segmentation method  

`input_image_type` ("stitched" | "retiled"): Must match segmentation mode  
- When `segmentation_file` is used: it defines the segmentation mask, but `input_image_type` still controls which raw image is loaded for overlay — these must match (e.g. stitched mask → `"stitched"` image).

`output_dir_prefix` (str | Path | None): Where outputs are saved (default = inside `input_dir`)  
`DAPI_ch` (int): DAPI channel (default = 4)  
`crop_coords` (tuple | None): `(y0, y1, x0, x1)` for zoom (default = full image)



#### Key behavior

- `segmentation_file` → fully controls which mask is used  
- default lookup → `{region}_{segmentation_method}_{input_image_type}_expanded.npz`  
- mask must match image shape  
- `input_dir` is always used to load the raw image  


#### Note

Only one region at a time: `region = "R1"` (not a list)

In [ ]:
input_dir = '/path/to/regions/'
region = 'R1'
segmentation_file = None


In [ ]:
SEG.inspect_and_work_with_segmentation(
    input_dir=input_dir,
    region=region,
    segmentation_method="cellpose",   # or "stardist", or "baysor"
    segmentation_file=segmentation_file,
    output_dir_prefix=None,
    input_image_type="retiled",      # or "retiled"
    DAPI_ch=4,
    crop_coords=None,
    
)